In [69]:
import torch
from torch import Tensor, nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [70]:
class NamesDataset(Dataset):
    def __init__(self, data_dir='names', max_len=20):
        self.countries: list[str] = [
            'Arabic', 'Chinese', 'Czech', 'Dutch', 'English',
            'French', 'German', 'Greek', 'Irish', 'Italian',
            'Japanese', 'Korean', 'Polish', 'Portuguese',
            'Russian', 'Scottish', 'Spanish', 'Vietnamese'
        ]
        self.country_to_idx: dict[str, int] = {
            c: i for i, c in enumerate(self.countries)}

        self.samples: list[tuple[str, int]] = []
        self.max_len = max_len

        data_path = Path(data_dir)
        for file_path in data_path.glob('*.txt'):
            country = file_path.stem
            if country not in self.country_to_idx:
                continue

            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    name = line.strip()
                    cleaned_name = []
                    for c in name:
                        if c.isalpha() and c != '\xa0':
                            cleaned_name.append(c)
                    name = ''.join(cleaned_name).strip()
                    if not name:
                        continue
                    self.samples.append(
                        (name, self.country_to_idx[country]))
        chars: set[str] = set()
        for name, _ in self.samples:
            for c in name:
                chars.add(c)
        self.char_to_idx: dict[str, int] = {
            c: i for i, c in enumerate(sorted(chars))}
        print(self.char_to_idx)
        self.vocab_size = len(self.char_to_idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name, country_idx = self.samples[idx]
        name_tensor: torch.Tensor = torch.zeros(
            self.max_len, len(self.char_to_idx))
        for i, c in enumerate(name[:self.max_len]):
            name_tensor[i, self.char_to_idx[c]] = 1

        country_tensor: torch.Tensor = torch.zeros(18)
        country_tensor[country_idx] = 1

        return name_tensor, country_tensor, country_idx, name

In [71]:
class NameClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_size) when batch_first=True
        _, (hidden, _) = self.rnn(x)
        # hidden: (1, batch, hidden_size)
        out = hidden[-1]  # (batch, hidden_size)
        
        out = self.fc(out)  # (batch, 18)
        return out

In [72]:
def train(model: NameClassifier, dataset: NamesDataset, train_loader: DataLoader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        epoch_loss = 0
        correct = 0
        total = 0
        for x, _, y, _ in train_loader:
            x: Tensor = x.to(device)
            y: Tensor = y.to(device)
            
            y_pred = model(x)
            loss: Tensor = criterion(y_pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pred = torch.argmax(y_pred, dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

            epoch_loss += loss.item()
        acc = correct / total
        print(
            f"Epoch {epoch+1}, Loss: {epoch_loss / total:.4f}, Acc: {acc:.4f}")

In [73]:
def predict(model: nn.Module, name: str, dataset: NamesDataset):
    model.eval()

    x = torch.zeros(1, dataset.max_len, dataset.vocab_size).to(device)
    print(x.shape)
    for i, c in enumerate(name[:dataset.max_len]):
        if c in dataset.char_to_idx:
            x[0, i, dataset.char_to_idx[c]] = 1

    with torch.no_grad():
        output = model(x)
        pred_idx = torch.argmax(output, dim=1).item()

    return dataset.countries[int(pred_idx)]

In [74]:
dataset = NamesDataset('names')
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
model = NameClassifier(
    input_size=dataset.vocab_size,
    hidden_size=128,
    num_classes=len(dataset.countries)).to(device)
train(model, dataset, train_loader)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 'a': 26, 'b': 27, 'c': 28, 'd': 29, 'e': 30, 'f': 31, 'g': 32, 'h': 33, 'i': 34, 'j': 35, 'k': 36, 'l': 37, 'm': 38, 'n': 39, 'o': 40, 'p': 41, 'q': 42, 'r': 43, 's': 44, 't': 45, 'u': 46, 'v': 47, 'w': 48, 'x': 49, 'y': 50, 'z': 51, 'Á': 52, 'É': 53, 'ß': 54, 'à': 55, 'á': 56, 'ã': 57, 'ä': 58, 'ç': 59, 'è': 60, 'é': 61, 'ê': 62, 'ì': 63, 'í': 64, 'ñ': 65, 'ò': 66, 'ó': 67, 'õ': 68, 'ö': 69, 'ù': 70, 'ú': 71, 'ü': 72, 'ą': 73, 'ł': 74, 'ń': 75, 'Ś': 76, 'Ż': 77, 'ż': 78}
Epoch 1, Loss: 0.0297, Acc: 0.4668
Epoch 2, Loss: 0.0277, Acc: 0.4838
Epoch 3, Loss: 0.0232, Acc: 0.5515
Epoch 4, Loss: 0.0211, Acc: 0.5952
Epoch 5, Loss: 0.0187, Acc: 0.6514
Epoch 6, Loss: 0.0173, Acc: 0.6773
Epoch 7, Loss: 0.0160, Acc: 0.6997
Epoch 8, Loss: 0.0151, Acc: 0.7160
Epoch 9, Loss: 0.0

In [75]:
predict(model, "Liu Ye", dataset)

torch.Size([1, 20, 79])


'Chinese'